# 03 — Two-Stage Reverse (path B)  — PP+HP 버전 (스태킹 다양성용)

기존 `ts_reverse.ipynb`(HP만 흔드는 노트북)의 **쌍둥이** — 여기에 **PP 6축을 Optuna trial 축에 추가**해서 같은 모델이라도 다른 전처리에서 학습한 OOF를 만든다 → 스태킹 base 다양성 ↑.

- PP 6축: `missing_threshold / corr_threshold / add_indicator / indicator_threshold / spatial_max_dist / post_impute_corr_threshold` ([pp_hp_strategy.md §3](../../../pp_hp_strategy.md)). 나머지는 PP_FIXED와 동일하게 고정. trial마다 `pp_hpo.make_cached_preprocess`로 전처리(LRU 캐시).
- HP·anchor·CV·후처리·산출물은 `ts_reverse.ipynb`와 동일. anchor에 PP_FIXED 값을 `pp_*` 키로 추가(corr 0.90->0.88).
- 출력 폴더만 다름: `4_output/.../pphp/`.

> ⚠ pp+hp는 trial마다 전처리(spatial impute 포함)를 다시 도므로 hp-only보다 느리다. `PP_CACHE_SIZE`로 캐시 보관 개수 조정.


## 1. 환경 설정 + import

Colab/Local 자동 감지. Colab 사용 시 `GDRIVE_MODELING_ID` 채울 것.

In [ ]:
import os, sys

# Google Drive 파일 ID들 — Colab에서 코드/데이터/모듈 zip을 자동으로 받아 풀 때 사용 (로컬은 무시)
GDRIVE_CODE_ID         = '1AD4PDBnDVjp-LSna6puB7qLnpBqB7j_I'   # code.zip = setup.py + utils/
GDRIVE_DATASET_ID      = '1yOUo0_wPLcuZBSJIK592b00YkSIlk4zO'   # dataset.zip = 원본 CSV 4개
GDRIVE_PREPROCESSING_ID = '1Rh0ByOS4Gama8XHuvY7KkOHo278H9YLr'  # preprocessing.zip = cleaning/outlier/scaling 등
GDRIVE_MODELING_ID     = '1Vrn5LBl611rWbag7d09LZH68_lfpu6wP'   # modeling.zip = 3_modeling/modules
GDRIVE_OUTPUT_ID       = '1ts73qEMmjX8cKIb-QeDQ-TMeyudFGWzs'  # 4_output.zip = 기존 실험 산출물 (RESUME용)
RESUME                 = True   # True=기존 optuna db에 trial 이어 붙임 / False=처음부터 (db 있으면 의도적 에러)

# Colab이면 필요한 zip들을 받아 풀고(이미 풀려 있으면 skip), 로컬이면 ../../../setup.py만
try:
    import google.colab
    if not os.path.exists('/content/project/setup.py'):
        os.system('pip install -q gdown')
        os.system(f'gdown {GDRIVE_CODE_ID} -O /content/code.zip')
        os.system('unzip -qo /content/code.zip -d /content/project')
        os.makedirs('/content/project/0_data', exist_ok=True)
        os.system(f'gdown {GDRIVE_DATASET_ID} -O /content/project/0_data/dataset.zip')
        os.system('unzip -qo /content/project/0_data/dataset.zip -d /content/project/0_data')
        os.remove('/content/project/0_data/dataset.zip')
    if not os.path.exists('/content/project/2_preprocessing/cleaning.py'):
        os.system(f'gdown {GDRIVE_PREPROCESSING_ID} -O /content/preprocessing.zip')
        os.system('unzip -qo /content/preprocessing.zip -d /content/project')
    if GDRIVE_MODELING_ID and not os.path.exists('/content/project/3_modeling/modules/zit.py'):
        os.system(f'gdown {GDRIVE_MODELING_ID} -O /content/modeling.zip')
        os.makedirs('/content/project/3_modeling', exist_ok=True)
        os.system('unzip -qo /content/modeling.zip -d /content/project/3_modeling')
    if RESUME and GDRIVE_OUTPUT_ID and not os.path.exists('/content/project/4_output/01_zit'):
        os.system(f'gdown {GDRIVE_OUTPUT_ID} -O /content/4_output.zip')
        os.system('unzip -qo /content/4_output.zip -d /content/project')
        os.remove('/content/4_output.zip')
    sys.path.insert(0, '/content/project')
    %run /content/project/setup.py
except ImportError:
    %run ../../../setup.py

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from utils.config import PROJECT_ROOT, SEED, TARGET_COL, KEY_COL, DIE_KEY_COL, OUTPUT_DIR
from utils.data import load_all, get_feat_cols, split_xs

# 전처리 모듈(2_preprocessing) 경로 + `from modules import ...` 가 3_modeling/modules를 찾게
PP_DIR = os.path.join(PROJECT_ROOT, '2_preprocessing')
if PP_DIR not in sys.path:
    sys.path.insert(0, PP_DIR)
MOD_DIR = os.path.join(PROJECT_ROOT, '3_modeling')
if MOD_DIR not in sys.path:
    sys.path.insert(0, MOD_DIR)

from modules import preprocess, hpo, postprocess, pp_hpo
from meta_features import add_meta_features

import lightgbm as lgb
import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner
from sklearn.model_selection import KFold

import logging, time
logging.getLogger('lightgbm').setLevel(logging.ERROR)
optuna.logging.set_verbosity(optuna.logging.WARNING)

print(f'PROJECT_ROOT = {PROJECT_ROOT}')
print(f'optuna v{optuna.__version__}')

## 2. 실험 설정

- `TS_REVERSE_ANCHOR` — 1차 ts-reverse-hpo-001 best HP (val=0.005709, test=0.008412)
- `TS_REVERSE_SEARCH` — narrow_around로 LGBM/w0 자동 산출 + categorical 4종 수동 추가
- categorical: `reg_objective ∈ {regression, poisson, tweedie_1.2, tweedie_1.5}`, `clf_scale_pos_weight ∈ {1.0, 1.5, 2.43, 3.5}` (1차 4종 그대로 탐색)

In [ ]:
# 실험 식별
EXP_ID = 'ts-reverse-pphp'
USER   = 'jh'

# Optuna 예산
N_TRIALS         = 1000
TIMEOUT_SEC      = 90 * 60 * 60  # 초 단위, None=무제한 (Colab 타임아웃 대비)
N_FOLDS          = 5
K_INNER          = 5    # outer fold 안에서 reg를 inner-OOF로 만들 때의 inner KFold 수 (unit 단위)
N_JOBS           = 4    # 모델 학습 병렬도
N_STARTUP_TRIALS = 40

# 출력 경로
OUT_DIR = os.path.join(OUTPUT_DIR, '03_two_stage', 'reverse', EXP_ID.split('-')[-1])
os.makedirs(OUT_DIR, exist_ok=True)
DB_PATH = os.path.join(OUT_DIR, f'optuna_{USER}_{EXP_ID}.db')

CLIP_Y_EXTREME = True   # train y의 max(1.0, 1건)를 두 번째 큰 값으로 clip

PP_CACHE_SIZE = 2   # cached_preprocess가 보관할 PP 조합 개수 (cleaned 3-split ≈ 1~1.5GB/개) — 메모리 보고 조정
# PP는 더 이상 고정값이 아니라 Optuna 탐색 6축 (pp_hpo.PP_SEARCH_CANDIDATES — pp_hp_strategy.md §3)

# anchor — 1차 ts-reverse best. w0 = "y=0 die에 줄 학습 가중치"(<1이면 0을 덜 중시), reg_objective/clf_scale_pos_weight는 따로 둠
TS_REVERSE_ANCHOR = {
    'n_estimators':      277,
    'learning_rate':     0.01152,
    'num_leaves':        476,
    'max_depth':         12,
    'min_child_samples': 22,
    'subsample':         0.989,
    'colsample_bytree':  0.750,
    'reg_alpha':         2.38e-07,
    'reg_lambda':        2.73e-07,
    'min_split_gain':    0.0789,
    'path_smooth':       31.80,
    'w0':                0.177,
}
ANCHOR_REG_OBJECTIVE = 'regression'
ANCHOR_CLF_SPW       = 2.43

# 탐색 공간 — LGBM HP + w0를 anchor 주변 ±30%로 자동 생성(narrow_around) + reg_objective/clf_scale_pos_weight를 수동 추가
TS_REVERSE_LOG_KEYS = {
    'learning_rate', 'reg_alpha', 'reg_lambda', 'min_split_gain', 'w0',
}
TS_REVERSE_SEARCH = hpo.narrow_around(
    TS_REVERSE_ANCHOR, log_keys=TS_REVERSE_LOG_KEYS,
    ratio=0.30, int_step_ratio=0.30,
)
# 물리적/논리적 범위 hard clip (확률은 [0,1], depth/leaves/min_child는 하한 등)
TS_REVERSE_SEARCH['subsample']['low']        = max(0.4, TS_REVERSE_SEARCH['subsample']['low'])
TS_REVERSE_SEARCH['subsample']['high']       = min(1.0, TS_REVERSE_SEARCH['subsample']['high'])
TS_REVERSE_SEARCH['colsample_bytree']['low']  = max(0.1, TS_REVERSE_SEARCH['colsample_bytree']['low'])
TS_REVERSE_SEARCH['colsample_bytree']['high'] = min(1.0, TS_REVERSE_SEARCH['colsample_bytree']['high'])
TS_REVERSE_SEARCH['max_depth']['low']         = max(3,   TS_REVERSE_SEARCH['max_depth']['low'])
TS_REVERSE_SEARCH['num_leaves']['low']        = max(8,   TS_REVERSE_SEARCH['num_leaves']['low'])
TS_REVERSE_SEARCH['min_child_samples']['low'] = max(5,   TS_REVERSE_SEARCH['min_child_samples']['low'])
TS_REVERSE_SEARCH['path_smooth']['low']       = max(0.0, TS_REVERSE_SEARCH['path_smooth']['low'])

# reg objective는 4종을 모두 탐색 (tweedie는 power 1.2/1.5 두 가지로 인코딩)
TS_REVERSE_SEARCH['reg_objective'] = {
    'type': 'cat',
    'choices': ['regression', 'poisson', 'tweedie_1.2', 'tweedie_1.5'],
}
# 분류기 클래스 가중치도 탐색 (음/양 비율 ~2.43 주변)
TS_REVERSE_SEARCH['clf_scale_pos_weight'] = {
    'type': 'float',
    'low': 1.5,
    'high': 4.0,
    'log': False,
}

print(f'EXP_ID={EXP_ID} | USER={USER}')
print(f'N_TRIALS={N_TRIALS} | N_FOLDS={N_FOLDS} | K_INNER={K_INNER} | N_JOBS={N_JOBS}')
print(f'OUT_DIR={OUT_DIR}')
print(f'DB_PATH={DB_PATH}')
print(f'\nTS_REVERSE_SEARCH ({len(TS_REVERSE_SEARCH)} HP):')
for k, spec in sorted(TS_REVERSE_SEARCH.items()):
    if spec['type'] == 'float':
        log_tag = ' log' if spec.get('log') else ''
        print(f'  {k:25s} float [{spec["low"]:.5g}, {spec["high"]:.5g}]{log_tag}')
    elif spec['type'] == 'int':
        print(f'  {k:25s} int   [{spec["low"]}, {spec["high"]}]')
    elif spec['type'] == 'cat':
        print(f'  {k:25s} cat   {spec["choices"]}')

## 3. 데이터 로드 + 전처리 캐시 준비 (PP는 Optuna 6축 — pp_hpo)


In [ ]:
xs, ys = load_all()
feat_cols = get_feat_cols(xs)
xs_dict = split_xs(xs)

# train y의 극단값(1.0, 1건)만 두 번째로 큰 값으로 clip — 학습 입력 안정화 (원본 ys는 보존)
ys_input = {k: v.copy() for k, v in ys.items()}
if CLIP_Y_EXTREME:
    y_raw = ys_input['train'][TARGET_COL]
    second_max = y_raw[y_raw < y_raw.max()].max()
    n_clipped = (y_raw >= 1.0).sum()
    ys_input['train'][TARGET_COL] = y_raw.clip(upper=second_max)
    print(f'[CLIP_Y_EXTREME] 1.0 -> {second_max:.6f} clip, {n_clipped}개 샘플')

# PP는 더 이상 1회 고정이 아니라 Optuna 6축 — trial마다 호출하는 캐시된 전처리 함수
# (preprocess.run(Stage0->cleaning->winsorize) + add_meta_features(position='raw', die_xy)를 PP 조합별로 캐시)
print('[PP 탐색 6축]')
for _k, _v in pp_hpo.PP_SEARCH_CANDIDATES.items():
    print(f'  {_k:28s} {_v}')
cached_prep = pp_hpo.make_cached_preprocess(
    xs, ys_input, feat_cols, xs_dict,
    position_mode='raw', use_die_xy=True, maxsize=PP_CACHE_SIZE, suppress_stdout=True,
)

# PP와 무관한 것들 (KEY_COL/DIE_KEY_COL/position은 전처리해도 안 바뀜) — 전처리 전 xs_dict에서 한 번만
uid_train_die = xs_dict['train'][KEY_COL].values
uid_val_die   = xs_dict['validation'][KEY_COL].values
uid_test_die  = xs_dict['test'][KEY_COL].values
y_train_unit_s = ys_input['train'].set_index(KEY_COL)[TARGET_COL]
y_val_unit_s   = ys_input['validation'].set_index(KEY_COL)[TARGET_COL]
y_test_unit_s  = ys_input['test'].set_index(KEY_COL)[TARGET_COL]
n_train_die = len(uid_train_die)
n_val_die   = len(uid_val_die)
n_test_die  = len(uid_test_die)
# Reverse Two-Stage는 die 단위 학습 → die에 unit health broadcast + 그 이진 라벨(y>0)
y_train_die_broadcast = pd.Series(uid_train_die).map(y_train_unit_s).values.astype(np.float64)
assert not pd.isna(y_train_die_broadcast).any(), 'unmapped train die y'
y_bin_die_broadcast = (y_train_die_broadcast > 0).astype(np.int32)

print(f'[데이터] train die={n_train_die:,}, unit train={len(y_train_unit_s):,}, val={len(y_val_unit_s):,}, test={len(y_test_unit_s):,}')
print(f'[cached_preprocess 준비] maxsize={PP_CACHE_SIZE} (X 행렬·feat_cols·xs_*는 PP에 따라 달라 objective/refit 안에서 cached_prep로 생성)')


## 4. K-fold split + helper + Optuna objective

- KFold는 **unit ID 단위 분할** (strategy_common §6) — 같은 unit의 4 die는 같은 fold
- helper inline: `_build_reg_params`, `_train_path_b(mode='innerOOF')`, `_mean_die_to_unit`, `_rmse_unit`
- inner KFold도 반드시 **unit 단위 분할** (die-level 분할 시 leakage)
- pruning: MedianPruner(n_warmup_steps=2)

In [ ]:
# unit ID 단위 K-fold (outer). 모든 trial 공유
unique_units = y_train_unit_s.index.values
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
FOLDS = list(kf.split(unique_units))


def _build_reg_params(hp, reg_obj):
    # LGBM HP에 회귀 objective를 끼워 넣음. 'tweedie_1.5' 같은 인코딩은 objective='tweedie' + variance_power=1.5로 풀어 줌
    p = dict(hp)
    if reg_obj.startswith('tweedie'):
        p['objective'] = 'tweedie'
        p['tweedie_variance_power'] = float(reg_obj.split('_')[1])
    else:
        p['objective'] = reg_obj
    return p


def _train_path_b(X_tr, y_tr_continuous, y_tr_bin, X_others,
                  hp, w0, reg_obj, clf_spw,
                  uid_tr, k_inner=5, seed=42):
    """Reverse Two-Stage의 한 outer fold를 학습하고, X_others 각각에 대해 (clf_prob, reg_pred, final=곱) 반환.

    구조:
      Stage 1 (reg): weighted MSE 회귀 — y=0 die에는 weight w0(<1), y>0 die에는 1. 이게 "0을 너무 중시하지 않게" 함.
      Stage 2 (clf): binary 분류 — 입력 X에 "Stage 1의 reg 예측" 한 컬럼을 더 붙여서 학습.
      누수 방지: clf 학습용 reg 예측은 outer-train 안에서 다시 inner-KFold(unit 단위) OOF로 만든다 (reg가 자기 train을 예측한 값을 쓰면 leakage).
      추론: X_others에는 outer-train 전체로 학습한 reg_full → clf 순으로 적용. final = clf_prob × reg_pred.
    """
    sw_full = np.where(y_tr_continuous == 0, w0, 1.0)   # y=0 die의 학습 가중치 = w0
    y_tr_log = y_tr_continuous  # (target_transform='none'이라 변환 안 함 — 변수명은 과거 흔적, log 의미 없음)
    reg_params = _build_reg_params(hp, reg_obj)

    # Stage 1 reg_full — outer-train 전체로 학습 (X_others 추론에 사용)
    reg_full = lgb.LGBMRegressor(**reg_params)
    reg_full.fit(X_tr, y_tr_log, sample_weight=sw_full)

    # clf 학습용 reg feature는 inner-OOF로 — outer-train을 inner KFold(unit 단위)로 나눠 각 검증분을 다른 fold로 학습한 reg가 예측
    unique_inner_units = np.unique(uid_tr)
    inner_kf = KFold(n_splits=k_inner, shuffle=True, random_state=seed)
    reg_train_oof_log = np.full(len(X_tr), np.nan)
    for itr_uidx, ivl_uidx in inner_kf.split(unique_inner_units):
        itr_units = unique_inner_units[itr_uidx]
        ivl_units = unique_inner_units[ivl_uidx]
        itr_die_mask = np.isin(uid_tr, itr_units)
        ivl_die_mask = np.isin(uid_tr, ivl_units)
        sw_inner = np.where(y_tr_continuous[itr_die_mask] == 0, w0, 1.0)
        reg_inner = lgb.LGBMRegressor(**reg_params)
        reg_inner.fit(
            X_tr[itr_die_mask],
            y_tr_log[itr_die_mask],
            sample_weight=sw_inner,
        )
        reg_train_oof_log[ivl_die_mask] = reg_inner.predict(X_tr[ivl_die_mask])
    assert not np.isnan(reg_train_oof_log).any(), 'inner OOF coverage bug'
    reg_train_y_for_clf = np.clip(reg_train_oof_log, 0.0, None)        # 음수 예측은 0으로
    X_tr_aug = np.hstack([X_tr, reg_train_y_for_clf.reshape(-1, 1)])   # X에 reg 예측 컬럼 추가

    # Stage 2 clf — 이진 분류(y>0 여부), 입력은 X + reg 예측 컬럼
    clf_params = dict(hp)
    clf_params['objective'] = 'binary'
    clf_params['scale_pos_weight'] = clf_spw
    clf = lgb.LGBMClassifier(**clf_params)
    clf.fit(X_tr_aug, y_tr_bin)

    # 추론: X_others 각각에 reg_full → (reg 예측을 컬럼으로 붙여) clf → final = prob × reg
    results = []
    for X_o in X_others:
        reg_log_o = reg_full.predict(X_o)
        reg_y_o   = np.clip(reg_log_o, 0.0, None)
        X_o_aug   = np.hstack([X_o, reg_y_o.reshape(-1, 1)])
        prob_o = np.clip(clf.predict_proba(X_o_aug)[:, 1], 0.0, 1.0)
        final_o = prob_o * reg_y_o
        results.append((prob_o, reg_y_o, final_o))
    return results, (reg_full, clf)


def _mean_die_to_unit(pred_die, uid_die):
    df = pd.DataFrame({KEY_COL: uid_die, 'pred': pred_die})
    return df.groupby(KEY_COL, sort=False)['pred'].mean().reset_index()


def objective(trial):
    t0 = time.time()
    # PP 6축 샘플 → 이 trial의 전처리 (cached_prep — 같은 PP 조합이면 캐시 hit, 재전처리 없음)
    _pp = pp_hpo.pp_search_space(trial)
    _xs_tr, _, _, _fcols, _ = cached_prep(_pp)
    X_train = _xs_tr[_fcols].values.astype(np.float64)
    trial.set_user_attr('pp_params', dict(_pp))
    sampled = hpo.sample_from_space(trial, TS_REVERSE_SEARCH)
    # 탐색 공간에서 w0 / reg_objective / clf_scale_pos_weight를 빼내면 나머지는 순수 LGBM HP
    w0       = sampled.pop('w0')
    reg_obj  = sampled.pop('reg_objective')
    clf_spw  = sampled.pop('clf_scale_pos_weight')
    hp = sampled

    hp['random_state']   = SEED
    hp['n_jobs']         = N_JOBS
    hp['verbose']        = -1
    hp['subsample_freq'] = 1   # 없으면 LGBM이 subsample 무시

    fold_oof_rmse = []
    oof_pred_unit = pd.Series(np.nan, index=y_train_unit_s.index, dtype=np.float64)

    for fold_idx, (tr_uidx, vl_uidx) in enumerate(FOLDS):
        tr_units = unique_units[tr_uidx]
        vl_units = unique_units[vl_uidx]
        tr_mask = np.isin(uid_train_die, tr_units)
        vl_mask = np.isin(uid_train_die, vl_units)

        # outer fold 학습 + 검증분 예측 (X_others에 검증 die만 넣음)
        results, _ = _train_path_b(
            X_train[tr_mask], y_train_die_broadcast[tr_mask], y_bin_die_broadcast[tr_mask],
            [X_train[vl_mask]],
            hp, w0, reg_obj, clf_spw,
            uid_tr=uid_train_die[tr_mask], k_inner=K_INNER, seed=SEED,
        )
        _, _, f_vl = results[0]   # (prob, reg, final) — final만 unit으로 집계
        unit_pred_df = _mean_die_to_unit(f_vl, uid_train_die[vl_mask])

        oof_pred_unit.loc[unit_pred_df[KEY_COL].values] = unit_pred_df['pred'].values
        y_vl = y_train_unit_s.loc[unit_pred_df[KEY_COL].values].values
        fold_rmse = float(np.sqrt(np.mean((unit_pred_df['pred'].values - y_vl) ** 2)))
        fold_oof_rmse.append(fold_rmse)

        # 누적 평균으로 pruning 판단
        avg = float(np.mean(fold_oof_rmse))
        trial.report(avg, step=fold_idx)
        if trial.should_prune():
            trial.set_user_attr('pruned_at_fold', fold_idx + 1)
            trial.set_user_attr('elapsed_sec', time.time() - t0)
            trial.set_user_attr('w0', w0)
            trial.set_user_attr('reg_objective', reg_obj)
            trial.set_user_attr('clf_scale_pos_weight', clf_spw)
            raise optuna.TrialPruned()

    if oof_pred_unit.isna().any():
        raise RuntimeError('OOF NaN — fold 누락')

    # 전체 OOF unit RMSE = trial 점수. w0/reg_obj/clf_spw는 best 복원용으로 user_attr에도 기록
    oof_rmse = float(np.sqrt(np.mean((oof_pred_unit.values - y_train_unit_s.values) ** 2)))
    elapsed = time.time() - t0
    trial.set_user_attr('elapsed_sec', elapsed)
    trial.set_user_attr('w0', w0)
    trial.set_user_attr('reg_objective', reg_obj)
    trial.set_user_attr('clf_scale_pos_weight', clf_spw)
    trial.set_user_attr('fold_oof_rmse', fold_oof_rmse)
    print(f'  trial #{trial.number}: oof={oof_rmse:.6f}, w0={w0:.3f}, reg={reg_obj}, spw={clf_spw}, elapsed={elapsed:.0f}s')
    return oof_rmse


print(f'fold split: {N_FOLDS} folds, unit 단위 분할, seed={SEED}')
print(f'inner KFold: K_INNER={K_INNER}, unit 단위')

## 5. Optuna study 생성 + anchor enqueue + optimize

- TPESampler(seed=None, multivariate=True, group=True) — strategy_common §4
- `enqueue_anchor`로 첫 trial은 1차 best HP 그대로

In [ ]:
sampler = TPESampler(
    seed=None,
    multivariate=True,
    group=True,
    n_startup_trials=N_STARTUP_TRIALS,
)
pruner = MedianPruner(n_startup_trials=N_STARTUP_TRIALS, n_warmup_steps=2)

study = optuna.create_study(
    study_name=EXP_ID,
    storage=f'sqlite:///{DB_PATH}',
    sampler=sampler,
    pruner=pruner,
    direction='minimize',
    load_if_exists=RESUME,
)

# anchor(=1차 best LGBM HP + w0 + reg_objective + clf_scale_pos_weight)를 trial 0으로 강제 — 이미 trial 있으면 skip
ANCHOR_FOR_ENQUEUE = dict(TS_REVERSE_ANCHOR)
ANCHOR_FOR_ENQUEUE['reg_objective']        = ANCHOR_REG_OBJECTIVE
ANCHOR_FOR_ENQUEUE['clf_scale_pos_weight'] = ANCHOR_CLF_SPW
ANCHOR_FOR_ENQUEUE.update({   # 1차 고정 전처리 값을 pp_* 키로 (corr 0.90은 후보 [0.80,0.84,0.88,0.92,0.96,0.98]에 없어 0.88)
    'pp_missing_threshold': 0.30, 'pp_corr_threshold': 0.88, 'pp_add_indicator': True,
    'pp_indicator_threshold': 0.05, 'pp_spatial_max_dist': 6.0, 'pp_post_impute_corr_threshold': 0.96,
})
if len(study.trials) == 0:
    hpo.enqueue_anchor(study, ANCHOR_FOR_ENQUEUE)
else:
    print(f'[enqueue skip] 기존 trial {len(study.trials)} 있음 — resume')

# 재현성 메타를 study에 박제
study_meta = {
    'exp_id': EXP_ID, 'user': USER, 'model': 'Reverse Two-Stage (path B, innerOOF)',
    'n_trials': N_TRIALS, 'n_folds': N_FOLDS, 'k_inner': K_INNER, 'n_jobs': N_JOBS,
    'pp_search_candidates': pp_hpo.PP_SEARCH_CANDIDATES,
    'anchor': TS_REVERSE_ANCHOR,
    'anchor_reg_objective': ANCHOR_REG_OBJECTIVE,
    'anchor_clf_spw': ANCHOR_CLF_SPW,
    'sampler': 'TPE seed=None multivariate group',
    'pruner':  f'MedianPruner n_startup={N_STARTUP_TRIALS} n_warmup=2',
    'CLIP_Y_EXTREME': CLIP_Y_EXTREME, 'SEED': int(SEED),
}
for k, v in study_meta.items():
    study.set_user_attr(k, str(v))

print(f'study: {study.study_name}, DB: {DB_PATH}')
print(f'기존 trial: {len(study.trials)}')

# HPO 실행 — trial 직렬(n_jobs=1)
t_start = time.time()
study.optimize(objective, n_trials=N_TRIALS, timeout=TIMEOUT_SEC, n_jobs=1, show_progress_bar=True)
print(f'\n[HPO 완료] 전체 {time.time()-t_start:.0f}s, total trials={len(study.trials)}')
print(f'  best OOF RMSE: {study.best_value:.6f}')

## 6. Best trial 정보 + anchor enqueue 검증

In [ ]:
best_trial = study.best_trial
best_params_full = best_trial.params   # LGBM HP + w0 + reg_objective + clf_scale_pos_weight
best_w0       = best_params_full['w0']
best_reg_obj  = best_params_full['reg_objective']
best_clf_spw  = best_params_full['clf_scale_pos_weight']
# refit에 넘길 순수 LGBM HP만 추림
hp_best = {
    k: v for k, v in best_params_full.items()
    if k not in ['w0', 'reg_objective', 'clf_scale_pos_weight'] and not str(k).startswith('pp_')
}

print(f'=== Best Trial #{best_trial.number} ===')
print(f'  OOF RMSE      : {best_trial.value:.6f}')
print(f'  best w0       : {best_w0:.4f}')
print(f'  best reg_obj  : {best_reg_obj}')
print(f'  best clf_spw  : {best_clf_spw}')
print(f'  elapsed       : {best_trial.user_attrs.get("elapsed_sec", 0):.0f}s')
for k, v in sorted(hp_best.items()):
    print(f'    {k}: {v}')

# trial 0이 anchor와 일치하는지 검증 (enqueue 정상 동작) — 숫자는 1e-9 허용오차, 문자열은 정확히
trial0 = study.trials[0]
def _eq(a, b, tol=1e-9):
    if isinstance(a, str) or isinstance(b, str):
        return a == b
    try:
        return abs(float(a) - float(b)) < tol
    except (TypeError, ValueError):
        return a == b
anchor_check = all(
    _eq(trial0.params.get(k), v) for k, v in ANCHOR_FOR_ENQUEUE.items()
)
print(f'\n[검증] trial 0 == anchor? {anchor_check}')

## 7. Best HP 5-fold refit (innerOOF) + die-level prob/reg/pred 캐쳐

In [ ]:
# best PP로 전처리 재실행 (best PP 조합은 보통 캐시에 이미 있음) → X 행렬·feat_cols·xs_* 재구성
_best_pp = pp_hpo.pp_params_from_best(best_params_full)
xs_train, xs_val, xs_test, feat_cols_clean, _eff_pp = cached_prep(_best_pp)
X_train = xs_train[feat_cols_clean].values.astype(np.float64)
X_val   = xs_val[feat_cols_clean].values.astype(np.float64)
X_test  = xs_test[feat_cols_clean].values.astype(np.float64)
print(f"[best PP] {_best_pp}")
print(f"[best PP 전처리 후 feat_cols] {len(feat_cols_clean)} | PP 캐시: {cached_prep.counters}")

# best HP로 모델 인자 보강
hp_refit = dict(hp_best)
hp_refit['random_state']   = SEED
hp_refit['n_jobs']         = N_JOBS
hp_refit['verbose']        = -1
hp_refit['subsample_freq'] = 1

# train OOF, val/test fold 평균 — prob / reg / pred(=prob·reg) 각각
oof_die_prob   = np.full(n_train_die, np.nan)
oof_die_reg    = np.full(n_train_die, np.nan)
oof_die_pred   = np.full(n_train_die, np.nan)

val_die_prob   = np.zeros(n_val_die)
val_die_reg    = np.zeros(n_val_die)
val_die_pred   = np.zeros(n_val_die)
test_die_prob  = np.zeros(n_test_die)
test_die_reg   = np.zeros(n_test_die)
test_die_pred  = np.zeros(n_test_die)

fold_models = []  # fold마다 (reg_full, clf) 튜플

print(f'=== Best HP 5-fold refit (Path B + innerOOF) ===')
t0 = time.time()
for fold_idx, (tr_uidx, vl_uidx) in enumerate(FOLDS):
    tr_units = unique_units[tr_uidx]
    vl_units = unique_units[vl_uidx]
    tr_mask = np.isin(uid_train_die, tr_units)
    vl_mask = np.isin(uid_train_die, vl_units)

    # X_others = [검증 die, val 전체, test 전체] → 한 번에 세 split 예측
    results, models = _train_path_b(
        X_train[tr_mask], y_train_die_broadcast[tr_mask], y_bin_die_broadcast[tr_mask],
        [X_train[vl_mask], X_val, X_test],
        hp_refit, best_w0, best_reg_obj, best_clf_spw,
        uid_tr=uid_train_die[tr_mask], k_inner=K_INNER, seed=SEED,
    )
    (p_vl, r_vl, f_vl), (p_v, r_v, f_v), (p_t, r_t, f_t) = results

    # 검증분은 OOF 자리에, val/test는 fold 평균 누적
    oof_die_prob[vl_mask] = p_vl
    oof_die_reg[vl_mask]  = r_vl
    oof_die_pred[vl_mask] = f_vl

    val_die_prob  += p_v / N_FOLDS
    val_die_reg   += r_v / N_FOLDS
    val_die_pred  += f_v / N_FOLDS
    test_die_prob += p_t / N_FOLDS
    test_die_reg  += r_t / N_FOLDS
    test_die_pred += f_t / N_FOLDS

    fold_models.append(models)
    print(f'  fold {fold_idx+1}/{N_FOLDS} done ({time.time()-t0:.0f}s)')

assert not np.isnan(oof_die_prob).any()
assert not np.isnan(oof_die_reg).any()
assert not np.isnan(oof_die_pred).any()
print(f'\n[refit 완료] die-level prob/reg/pred 캐쳐 OK')

## 8. 후처리 — 집계 8 + position Optuna + π threshold + zero_clip(log)

- 분류 threshold (§9): **APPLY** — Reverse는 die-level prob을 cut 안 했으므로 후처리에서 unit 평균 prob에 threshold 탐색
- 집계 다양성 (§10): 8종
- Position 가중치 (§11): Optuna sub-study 50 trial
- zero_clip (§12): original space 비교 (TARGET_TRANSFORM='none', strategy_common §24)

In [ ]:
# 후처리: die→unit 집계 8종 best + position 가중평균(Optuna 50t) + π threshold + zero_clip — 각 단계 val 개선 시만 채택.
# Reverse는 die-level에서 prob을 cut 하지 않으므로 use_pi_threshold=True (후처리에서 unit 평균 prob에 threshold 탐색).
# target_transform='none'이라 zero_clip 비교는 원본 공간(log_space=False).
pp_res = postprocess.tune_and_apply(
    xs_train, xs_val, xs_test,
    die_pred_train=oof_die_pred,
    die_pred_val=val_die_pred,
    die_pred_test=test_die_pred,
    die_pi_train=oof_die_prob,
    die_pi_val=val_die_prob,
    die_pi_test=test_die_prob,
    y_train_unit=ys_input['train'],
    use_pi_threshold=True,
    agg_methods=postprocess.AGG_METHODS,
    zero_clip_log_space=False,
    position_method='optuna',
    position_optuna_n_trials=50,
)

print(f'\n[Postprocess]')
print(f'  best_agg            : {pp_res["best_agg"]}')
print(f'  pos_weights         : {pp_res["pos_weights"]}')
print(f'  best_pi_threshold   : {pp_res["best_pi_threshold"]}')
_bzc = pp_res["best_zero_clip"]
print(f'  best_zero_clip      : {_bzc:.4f}' if _bzc is not None else '  best_zero_clip      : None')
print(f'  position_method     : {pp_res["position_method"]}')
print(f'  train_rmse          : {pp_res["train_rmse"]:.6f}')

# 후처리 적용본의 val/test unit RMSE 직접 계산
if pp_res.get('final_val_unit') is not None:
    _val_pred = pp_res['final_val_unit'].set_index(KEY_COL)['pred'].loc[y_val_unit_s.index]
    val_rmse  = float(np.sqrt(np.mean((_val_pred.values  - y_val_unit_s.values)  ** 2)))
    print(f'  val_rmse            : {val_rmse:.6f}')
if pp_res.get('final_test_unit') is not None:
    _test_pred = pp_res['final_test_unit'].set_index(KEY_COL)['pred'].loc[y_test_unit_s.index]
    test_rmse  = float(np.sqrt(np.mean((_test_pred.values - y_test_unit_s.values) ** 2)))
    print(f'  test_rmse           : {test_rmse:.6f}')

print(f'  agg_rmses           : {pp_res["agg_rmses"]}')

## 9. 산출물 9개 저장 (strategy_common §15)

best_params.json + fold_models.pkl + 6 CSV (die ×3 + unit ×3) + optuna_*.db

In [ ]:
import json, pickle, hashlib

# 1) fold_models.pkl — fold마다 (reg_full, clf) 튜플 + feature 이름
with open(os.path.join(OUT_DIR, 'fold_models.pkl'), 'wb') as f:
    pickle.dump({
        'fold_models':   fold_models,    # list of (reg, clf)
        'feature_names': feat_cols_clean,
        'model_name':    'ts_reverse',
        'n_folds':       N_FOLDS,
    }, f)

# 2) best_params.json — 재현성 메타 + train unit 목록 해시(다른 단계 OOF와 분할 일치 검증용)
uid_arr = ys_input['train'][KEY_COL].unique()
unit_ids_hash = hashlib.sha1(','.join(map(str, uid_arr)).encode()).hexdigest()

best_meta = {
    'exp_id':                EXP_ID,
    'model_name':            'ts_reverse',
    'best_trial_number':     best_trial.number,
    'best_oof_rmse':         float(best_trial.value),
    'best_params_resolved':  hp_refit,
    'best_w0':               float(best_w0),
    'best_reg_objective':    best_reg_obj,
    'best_clf_scale_pos_weight': float(best_clf_spw),
    'feature_names':         feat_cols_clean,
    'n_features':            len(feat_cols_clean),
    'n_folds':               N_FOLDS,
    'k_inner':               K_INNER,
    'unit_ids_hash':         unit_ids_hash,
    'n_units_train':         int(len(uid_arr)),
    'effective_pp_params':   _eff_pp,
    'best_pp_params':        _best_pp,
    'study_meta':            study_meta,
    'postprocess': {
        'best_agg':            pp_res['best_agg'],
        'pos_weights':         pp_res['pos_weights'].tolist() if pp_res['pos_weights'] is not None else None,
        'best_pi_threshold':   (float(pp_res['best_pi_threshold'])
                                if pp_res['best_pi_threshold'] is not None else None),
        'best_zero_clip':      float(pp_res['best_zero_clip']) if pp_res['best_zero_clip'] is not None else None,
        'zero_clip_log_space': pp_res['zero_clip_log_space'],
        'position_method':     pp_res['position_method'],
        'agg_rmses':           {k: float(v) for k, v in pp_res['agg_rmses'].items()},
        'train_rmse':          float(pp_res['train_rmse']),
    },
}
with open(os.path.join(OUT_DIR, 'best_params.json'), 'w', encoding='utf-8') as f:
    json.dump(best_meta, f, indent=2, ensure_ascii=False, default=str)

# 3-5) die-level CSV — prob / reg / pred(=prob·reg) + health
def _build_die_df(uid, die_id, position, prob, reg, pred, y_unit):
    df = pd.DataFrame({
        KEY_COL: uid, DIE_KEY_COL: die_id, 'position': position,
        'prob': prob, 'reg': reg, 'pred': pred,
    })
    if y_unit is not None:
        df[TARGET_COL] = df[KEY_COL].map(y_unit)
    return df

_build_die_df(
    uid_train_die, xs_train[DIE_KEY_COL].values, xs_train['position'].values,
    oof_die_prob, oof_die_reg, oof_die_pred, y_train_unit_s,
).to_csv(os.path.join(OUT_DIR, 'oof_die.csv'), index=False)
_build_die_df(
    uid_val_die, xs_val[DIE_KEY_COL].values, xs_val['position'].values,
    val_die_prob, val_die_reg, val_die_pred, y_val_unit_s,
).to_csv(os.path.join(OUT_DIR, 'val_die.csv'), index=False)
_build_die_df(
    uid_test_die, xs_test[DIE_KEY_COL].values, xs_test['position'].values,
    test_die_prob, test_die_reg, test_die_pred, y_test_unit_s,
).to_csv(os.path.join(OUT_DIR, 'test_die.csv'), index=False)

# 6-8) unit-level CSV — 후처리 적용본 + health
def _build_unit_df(unit_pred_df, y_unit):
    out = unit_pred_df.copy()
    out[TARGET_COL] = out[KEY_COL].map(y_unit)
    return out

_build_unit_df(pp_res['final_train_unit'], y_train_unit_s).to_csv(os.path.join(OUT_DIR, 'oof_unit.csv'),  index=False)
_build_unit_df(pp_res['final_val_unit'],   y_val_unit_s  ).to_csv(os.path.join(OUT_DIR, 'val_unit.csv'),  index=False)
_build_unit_df(pp_res['final_test_unit'],  y_test_unit_s ).to_csv(os.path.join(OUT_DIR, 'test_unit.csv'), index=False)

# 9) optuna_*.db는 study.optimize가 자동 저장

# 저장된 파일 목록
print(f'\n저장 완료: {OUT_DIR}')
for fn in sorted(os.listdir(OUT_DIR)):
    sz = os.path.getsize(os.path.join(OUT_DIR, fn)) / 1024
    print(f'  {fn:30s}  {sz:10,.1f} KB')

# Colab이면 산출물을 zip으로 묶어 로컬 PC로 다운로드
try:
    import google.colab
    from google.colab import files
    import shutil
    _zip = shutil.make_archive(os.path.join('/content', f'ts_reverse_{EXP_ID}_outputs'), 'zip', OUT_DIR)
    print(f'\n[zip 생성] {_zip} ({os.path.getsize(_zip)/1024:.1f} KB)')
    try:
        files.download(_zip)
    except Exception as _e:
        from IPython.display import FileLink, display
        print(f'[files.download 실패: {_e}] 아래 링크 클릭')
        display(FileLink(_zip))
except ImportError:
    pass